# Liberman synapse comparison — classical (linear + trees)

Liberman **scenario C** only. Runs configuration panel **T1, T3–T6** (RF + XGB each) and **L7–L8** (OLS). **strain_binary** is in Stage 2 synapse/long features only (0 = CBA/CaJ, 1 = C57BL/6J); Stage 1 noise clf does not use strain. Wide tree/OLS stage-2 features use **50/60/70/80 dB** pivoted columns only. Long stage-2 (T1, T3) uses all SPL rows.

Data lineage: [`abr_wide_long_comparison.ipynb`](abr_wide_long_comparison.ipynb). Exports: `figures/cache/liberman_classical_comparison.parquet`, `liberman_best_tree_config.json`. No plots here — synthesis elsewhere.

| ID | Spec |
|----|------|
| T1 | Long, no label |
| T3 | Long, noise_pred |
| T4 | Wide, no label |
| T5 | Wide, noise_pred |
| T6 | Wide, true noise_cat (animal-level) |
| L7 | Wide OLS amp 80 dB |
| L8 | Wide OLS full + noise_pred |

In [1]:
import json
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import display

import utils.liberman_classical as lc
import utils.nn_stage2_data as nn2d
import utils.nn_stage2 as nn2

importlib.reload(nn2d)
importlib.reload(nn2)
importlib.reload(lc)

from utils.liberman_classical import (
    EXCLUDE_S2_EXTRA,
    animal_noise_series,
    attach_animal_noise_cat,
    derive_comparisons,
    diagnose_wide_noise_lift,
    export_artifacts,
    export_noise_diagnostic,
    export_t5_stage2_hp,
    liberman_feature_lists,
    run_config_panel,
)
from utils.nn_colab_export import DEFAULT_OUT
from utils.nn_colab_train import run_liberman_nn_hp_comparison
from utils.nn_stage2_data import (
    STAGE_SPL_LEVELS,
    load_nn_stage2_data,
    splits_for_long_stage2,
)

In [2]:
data = load_nn_stage2_data(join_io_features=False)
sp = splits_for_long_stage2(data)

lib_tr = sp["lib_train"].copy()
lib_te = sp["lib_test"].copy()
lib_long_tr = sp["lib_long_train"].copy()
lib_long_te = sp["lib_long_test"].copy()

_animal_noise = animal_noise_series(data.orig_lib)
lib_tr = attach_animal_noise_cat(lib_tr, _animal_noise)
lib_te = attach_animal_noise_cat(lib_te, _animal_noise)
lib_long_tr = attach_animal_noise_cat(lib_long_tr, _animal_noise)
lib_long_te = attach_animal_noise_cat(lib_long_te, _animal_noise)

feats = liberman_feature_lists(data.reformatted_orig, data.common_cols)

assert lib_tr["noise_cat"].isin([0, 1]).all()
assert lib_tr.groupby("animal_id")["noise_cat"].nunique().max() == 1

print(f"Wide tree SPL levels: {list(STAGE_SPL_LEVELS)} dB")
print(
    f"Liberman wide train/test: {len(lib_tr)} / {len(lib_te)} rows | "
    f"long train/test (all SPL): {len(lib_long_tr)} / {len(lib_long_te)}"
)
print(
    f"Synapse wide num/log: {len(feats['syn_num'])} / {len(feats['syn_log'])}"
)
print(
    f"Excluded extras present in syn_num: {set(feats['syn_num']) & EXCLUDE_S2_EXTRA}"
)

Wide tree SPL levels: [50, 60, 70, 80] dB
Liberman wide train/test: 491 / 125 rows | long train/test (all SPL): 9751 / 2436
Synapse wide num/log: 13 / 29
Excluded extras present in syn_num: set()


In [3]:
results = run_config_panel(
    lib_tr, lib_te, lib_long_tr, lib_long_te, feats, verbose=True
)
assert len(results) == 12, f"expected 12 rows, got {len(results)}"
display(results.sort_values(["config_id", "model"]))


=== T1 RF (long, noise=none) ===
  [T1-RF]  S1 wide train: 491 | long train: 9751 rows
           RF  — val acc (row): 0.702 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal): 0.952, AUC: 0.909
           Synapse reg — CV R²: 0.220, test R² (animal×freq): 0.317, RMSE: 3.030

=== T1 XGB (long, noise=none) ===
  [T1-XGB]  S1/S2 wide train: 491 | long train: 9751 rows
           RF  — val acc (row): 0.702 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal): 0.952, AUC: 0.909
           Synapse XGB — CV R²: 0.220, test R² (animal×freq): 0.331, RMSE: 2.998

=== T3 RF (long, noise=predicted) ===
  [T3-RF]  S1 wide train: 491 | long train: 9751 rows
           RF  — val acc (row): 0.702 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal): 0.952, AUC: 0.909
           Synapse reg — CV R²: 0.476, test R² (animal×freq): 0.597, RMSE: 2.326

=== T3 XGB (long, noise=predicted) ===
  [T3-XGB]  S1/S2 wi

,config_id,model,format,noise_label,r2_test,rmse_test
10,L7,OLS,wide,none,0.228920,3.090075
11,L8,OLS,wide,predicted,0.438487,2.636934
0,T1,RF,long,none,0.316922,3.030018
1,T1,XGB,long,none,0.331368,2.997807
2,T3,RF,long,predicted,0.597359,2.326316
3,T3,XGB,long,predicted,0.559444,2.433382
4,T4,RF,wide,none,0.486302,2.522162
5,T4,XGB,wide,none,0.469613,2.562805
6,T5,RF,wide,predicted,0.669408,2.023323
7,T5,XGB,wide,predicted,0.651600,2.077106


In [10]:
# Export T5 wide RF/XGB best_params for Section 5.3 scenario C (skip if JSON exists)
_hp_out = Path("figures/cache/stage2_best_hp/liberman_t5_sklearn.json")
if not _hp_out.is_file():
    export_t5_stage2_hp(lib_tr, lib_te, feats, out_path=_hp_out)
    print("Wrote", _hp_out)
else:
    print("Skip T5 HP export — already present:", _hp_out)

Wrote figures/cache/stage2_best_hp/liberman_t5_sklearn.json


In [5]:
comparisons = derive_comparisons(
    results,
    wide_train=lib_tr,
    wide_test=lib_te,
    long_train=lib_long_tr,
    long_test=lib_long_te,
    feats=feats,
)
display(comparisons)

for model in ("RF", "XGB"):
    q3 = comparisons[
        (comparisons["question"] == "Q3_format")
        & (comparisons["model"] == model)
    ]
    if not q3.empty:
        d = q3.iloc[0]["delta_r2"]
        fmt = "wide" if d > 0 else "long"
        print(f"  Q3 best format ({model}): {fmt} (delta_r2 T4-T1 = {d:.3f})")

,question,model,config_a,config_b,r2_a,r2_b,delta_r2,rmse_a,rmse_b,delta_rmse,...,mean_sq_err_b,mean_sq_err_diff_a_minus_b,f_stat,f_pvalue,variances_unequal,t_test,t_stat,t_pvalue,significant_at_alpha,alpha
0,Q3_format,RF,T1,T4,0.316922,0.486302,0.169380,3.030018,2.522162,-0.507856,...,6.361302,2.819707,2.157682,0.000024,True,welch,1.710541,0.088584,False,0.05
1,Q1_noise,RF,T4,T5,0.486302,0.669408,0.183106,2.522162,2.023323,-0.498839,...,4.093835,2.267467,1.848127,0.000708,True,welch,1.968979,0.050168,False,0.05
2,Q1_noise_long,RF,T1,T3,0.316922,0.597359,0.280438,3.030018,2.326316,-0.703702,...,5.411747,3.769262,2.030109,0.000098,True,welch,2.264159,0.024529,True,0.05
3,Q2_oracle,RF,T5,T6,0.669408,0.663359,-0.006049,2.023323,2.041750,0.018428,...,4.168745,-0.074910,1.109166,0.564903,False,paired,-0.579392,0.563376,False,0.05
4,Q3_format,XGB,T1,T4,0.331368,0.469613,0.138246,2.997807,2.562805,-0.435003,...,6.567969,2.418881,1.905663,0.000381,True,welch,1.474979,0.141610,False,0.05
5,Q1_noise,XGB,T4,T5,0.469613,0.651600,0.181986,2.562805,2.077106,-0.485699,...,4.314368,2.253601,1.121257,0.524921,False,paired,3.323807,0.001168,True,0.05
6,Q1_noise_long,XGB,T1,T3,0.331368,0.559444,0.228077,2.997807,2.433382,-0.564425,...,5.921350,3.065500,1.700392,0.003360,True,welch,1.831611,0.068290,False,0.05
7,Q2_oracle,XGB,T5,T6,0.651600,0.664509,0.012909,2.077106,2.038262,-0.038844,...,4.154511,0.159856,1.296275,0.149900,False,paired,0.576240,0.565497,False,0.05


  Q3 best format (RF): wide (delta_r2 T4-T1 = 0.169)
  Q3 best format (XGB): wide (delta_r2 T4-T1 = 0.138)


## Wide noise diagnostic (T4/T5/T6)

Stage-1 test errors vs stage-2 under-using ``noise_preds``. Also audits **train** ``noise_preds`` vs ``noise_cat`` (T5 vs T6).

In [6]:
cache_dir = Path("figures/cache")
for _model in ("RF", "XGB"):
    _anim, _train, _summary = diagnose_wide_noise_lift(
        lib_tr, lib_te, feats, model=_model
    )
    display(_anim.sort_values("pred_correct"))
    display(_train.sort_values("pred_correct"))
    print(json.dumps(_summary, indent=2))
    export_noise_diagnostic(
        _anim, _summary, cache_dir, model=_model, train_animal_tbl=_train
    )

,animal_id,noise_cat,noise_pred,noise_prob,pred_correct,n_rows,synapses,mse_T4,mse_T5,mse_T6,delta_mse_T5_minus_T4,delta_mse_T6_minus_T5
3,WPZ113,1,0,0.366657,False,6,14.898821,2.105476,2.715216,0.727980,0.609740,-1.987236
0,WPZ100,0,0,0.514810,True,6,15.892142,3.109071,1.178543,1.230773,-1.930528,0.052230
18,WPZ91,0,0,0.387516,True,6,13.842553,3.619714,2.312323,2.191411,-1.307391,-0.120912
17,WPZ83,0,0,0.442879,True,6,15.269309,1.500277,0.947918,0.851945,-0.552360,-0.095972
16,WPZ67,0,0,0.373788,True,6,14.620126,8.265631,9.743785,9.968619,1.478154,0.224833
15,WPZ62,1,1,0.591207,True,6,15.440508,4.763872,1.917976,1.971388,-2.845896,0.053412
14,WPZ51,1,1,0.676019,True,6,15.213716,5.325448,5.907413,5.277755,0.581965,-0.629658
13,WPZ179,1,1,0.744833,True,5,12.780952,9.570364,7.869923,7.818306,-1.700441,-0.051617
12,WPZ171,1,1,0.622090,True,6,15.559913,7.159306,4.636648,4.707894,-2.522657,0.071246
11,WPZ165,0,0,0.403734,True,6,14.892857,6.560796,0.948602,1.090426,-5.612194,0.141824


,animal_id,noise_cat,noise_pred,n_rows,pred_correct
10,WPZ123,1,0,6,False
59,WPZ60,0,1,6,False
8,WPZ117,1,0,6,False
61,WPZ66,1,0,6,False
32,WPZ155,1,0,5,False
...,...,...,...,...,...
25,WPZ144,1,1,5,True
24,WPZ143,1,1,6,True
23,WPZ142,0,0,6,True
42,WPZ169,1,1,6,True


{
  "model": "RF",
  "stage1": {
    "n_test_animals": 21,
    "n_correct": 20,
    "n_wrong": 1,
    "accuracy": 0.9523809523809523,
    "wrong_animal_ids": [
      "WPZ113"
    ],
    "threshold": 0.5557642115606013,
    "test_auc": 0.9090909090909091,
    "non_test_train_animal_accuracy": 0.9404761904761905
  },
  "train_noise_labels": {
    "n_train_animals": 84,
    "n_train_wide_rows": 491,
    "noise_preds_nan_rows": 0,
    "row_match_rate": 0.9409368635437881,
    "animal_match_rate": 0.9404761904761905,
    "n_mismatched_animals": 5,
    "mismatched_animal_ids": [
      "WPZ117",
      "WPZ123",
      "WPZ155",
      "WPZ60",
      "WPZ66"
    ]
  },
  "n_test_wide_rows": 125,
  "n_rows_correct_animals": 119,
  "n_rows_wrong_animals": 6,
  "r2_all_rows": {
    "T4": 0.48630230416669007,
    "T5": 0.6694083180806893,
    "T5_oracle_train_labels": 0.6550681955210417,
    "T6": 0.6633590911749419,
    "T5_oracle_test_labels": 0.6740614611105678
  },
  "r2_correct_animals_only": {

,animal_id,noise_cat,noise_pred,noise_prob,pred_correct,n_rows,synapses,mse_T4,mse_T5,mse_T6,delta_mse_T5_minus_T4,delta_mse_T6_minus_T5
3,WPZ113,1,0,0.366657,False,6,14.898821,1.567500,2.181402,1.030775,0.613901,-1.150627
0,WPZ100,0,0,0.514810,True,6,15.892142,2.732028,1.460792,1.336161,-1.271236,-0.124631
18,WPZ91,0,0,0.387516,True,6,13.842553,5.241535,2.349287,2.204886,-2.892248,-0.144401
17,WPZ83,0,0,0.442879,True,6,15.269309,1.960763,1.136564,1.060540,-0.824199,-0.076024
16,WPZ67,0,0,0.373788,True,6,14.620126,7.891658,10.452748,10.786259,2.561091,0.333510
15,WPZ62,1,1,0.591207,True,6,15.440508,4.783014,2.992822,1.929365,-1.790191,-1.063457
14,WPZ51,1,1,0.676019,True,6,15.213716,2.720702,6.032294,5.564467,3.311592,-0.467827
13,WPZ179,1,1,0.744833,True,5,12.780952,9.286383,7.789105,7.906759,-1.497279,0.117654
12,WPZ171,1,1,0.622090,True,6,15.559913,6.719047,2.722713,2.889500,-3.996335,0.166787
11,WPZ165,0,0,0.403734,True,6,14.892857,7.803043,1.553277,1.855240,-6.249766,0.301963


,animal_id,noise_cat,noise_pred,n_rows,pred_correct
10,WPZ123,1,0,6,False
59,WPZ60,0,1,6,False
8,WPZ117,1,0,6,False
61,WPZ66,1,0,6,False
32,WPZ155,1,0,5,False
...,...,...,...,...,...
25,WPZ144,1,1,5,True
24,WPZ143,1,1,6,True
23,WPZ142,0,0,6,True
42,WPZ169,1,1,6,True


{
  "model": "XGB",
  "stage1": {
    "n_test_animals": 21,
    "n_correct": 20,
    "n_wrong": 1,
    "accuracy": 0.9523809523809523,
    "wrong_animal_ids": [
      "WPZ113"
    ],
    "threshold": 0.5557642115606013,
    "test_auc": 0.9090909090909091,
    "non_test_train_animal_accuracy": 0.9404761904761905
  },
  "train_noise_labels": {
    "n_train_animals": 84,
    "n_train_wide_rows": 491,
    "noise_preds_nan_rows": 0,
    "row_match_rate": 0.9409368635437881,
    "animal_match_rate": 0.9404761904761905,
    "n_mismatched_animals": 5,
    "mismatched_animal_ids": [
      "WPZ117",
      "WPZ123",
      "WPZ155",
      "WPZ60",
      "WPZ66"
    ]
  },
  "n_test_wide_rows": 125,
  "n_rows_correct_animals": 119,
  "n_rows_wrong_animals": 6,
  "r2_all_rows": {
    "T4": 0.4696132607639858,
    "T5": 0.6515995287816416,
    "T5_oracle_train_labels": 0.6571959037156995,
    "T6": 0.6645085037611955,
    "T5_oracle_test_labels": 0.6481238245195681
  },
  "r2_correct_animals_only": {

## Export Colab GPU pack (NN HP tuning)

Preprocessed long-format tensors for [`abr_nn_stage2_colab.ipynb`](abr_nn_stage2_colab.ipynb): tabular features (fitted scaler on Train rows), Wave-I / full-wave arrays, Stage-1 `noise_preds`, and train / validate / test splits. Upload `figures/cache/nn_colab_liberman/` to Google Drive before running the Colab notebook.

In [7]:
from utils.nn_colab_export import DEFAULT_OUT, export_liberman_nn_colab_pack

colab_pack_dir = export_liberman_nn_colab_pack(DEFAULT_OUT)
print(f"Upload to Colab: {colab_pack_dir.resolve()}")

           Selected RF  — fit acc (animal@cal): 0.971 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal@cal): 0.952 | test acc@0.5: 0.810 | AUC: 0.909
Exported Colab pack → /Users/nowaki027/MSDS/Practicum/figures/cache/nn_colab_liberman
  train 7869 rows, 68 animals | validate 1882 | test 2436
  tabular dim=14, wave_full_len=201
Upload to Colab: /Users/nowaki027/MSDS/Practicum/figures/cache/nn_colab_liberman


## NN Stage 2 (long) — Liberman HP comparison

Full HP grids for three architectures (same grids as Colab export manifest):

- **N1 / `mlp`** — tabular only (72 configs)
- **N2 / `cnn`** — Wave I + tabular
- **N3 / `cnn_full`** — full 0–8 ms waveform + tabular

Uses the exported Colab pack above (noise_pred, **strain_binary** when present). **Downstream synthesis / scenario C uses MLP only** — this panel confirms MLP remains best.

In [8]:
cache_dir = Path("figures/cache")
nn_out = DEFAULT_OUT / "results"
summary, nn_results = run_liberman_nn_hp_comparison(
    DEFAULT_OUT, nn_out, verbose=True
)
display(nn_results.sort_values("r2_test", ascending=False))
best = nn_results.loc[nn_results["r2_test"].idxmax()]
print(
    f"Best NN: {best['model']} ({best['config_id']}) "
    f"R²={best['r2_test']:.3f} RMSE={best['rmse_test']:.3f} — production path uses MLP only"
)
nn_results.to_parquet(
    cache_dir / "liberman_nn_hp_comparison.parquet", index=False
)
summary.to_parquet(cache_dir / "liberman_nn_hp_summary.parquet", index=False)
print("Wrote", cache_dir / "liberman_nn_hp_comparison.parquet")


=== HP tuning: mlp ===
  [mlp 1/72] hidden=(64,) dropout=0.1 lr=0.0001 wd=0.0001 rmse=6.0735
  [mlp 2/72] hidden=(64,) dropout=0.1 lr=0.0001 wd=0.001 rmse=6.0742
  [mlp 3/72] hidden=(64,) dropout=0.1 lr=0.001 wd=0.0001 rmse=2.4553
  [mlp 4/72] hidden=(64,) dropout=0.1 lr=0.001 wd=0.001 rmse=2.4563
  [mlp 5/72] hidden=(64,) dropout=0.1 lr=0.005 wd=0.0001 rmse=2.3886
  [mlp 6/72] hidden=(64,) dropout=0.1 lr=0.005 wd=0.001 rmse=2.3853
  [mlp 7/72] hidden=(64,) dropout=0.2 lr=0.0001 wd=0.0001 rmse=6.0957
  [mlp 8/72] hidden=(64,) dropout=0.2 lr=0.0001 wd=0.001 rmse=6.0965
  [mlp 9/72] hidden=(64,) dropout=0.2 lr=0.001 wd=0.0001 rmse=2.4688
  [mlp 10/72] hidden=(64,) dropout=0.2 lr=0.001 wd=0.001 rmse=2.4689
  [mlp 11/72] hidden=(64,) dropout=0.2 lr=0.005 wd=0.0001 rmse=2.3947
  [mlp 12/72] hidden=(64,) dropout=0.2 lr=0.005 wd=0.001 rmse=2.3957
  [mlp 13/72] hidden=(64,) dropout=0.3 lr=0.0001 wd=0.0001 rmse=6.1236
  [mlp 14/72] hidden=(64,) dropout=0.3 lr=0.0001 wd=0.001 rmse=6.1244
  [mlp

,config_id,model,format,noise_label,r2_test,rmse_test
2,N1,mlp,long,predicted,0.456319,2.703224
0,N2,cnn,long,predicted,0.351978,2.951242
1,N3,cnn_full,long,predicted,0.250285,3.174374


Best NN: mlp (N1) R²=0.456 RMSE=2.703 — production path uses MLP only
Wrote figures/cache/liberman_nn_hp_comparison.parquet


In [9]:
cache_dir = Path("figures/cache")
pq_path, json_path, comp_path = export_artifacts(
    results, comparisons, cache_dir
)
print(f"Wrote {pq_path}")
print(f"Wrote {comp_path}")
print(f"Wrote {json_path}")

Wrote figures/cache/liberman_classical_comparison.parquet
Wrote figures/cache/liberman_classical_comparisons.parquet
Wrote figures/cache/liberman_best_tree_config.json
